<a href="https://colab.research.google.com/github/pnperl/Heiken-Ashi-Screener-HA-reversal/blob/main/Copy_of_Heiken_Ashi_Screener_HA_reversal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance pandas numpy requests python-dotenv -q

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TRADING BOT — GOOGLE COLAB VERSION                 ║
# ╚══════════════════════════════════════════════════════════════╝
#
# HOW TO USE IN COLAB:
#   1. Run CELL 1 first (installs packages)
#   2. Add TELEGRAM_TOKEN and TELEGRAM_CHAT_ID in Colab Secrets
#      (🔑 key icon on left sidebar)
#   3. Run CELL 2 (all functions)
#   4. Run CELL 3 (main loop — bot starts)
#
# ✏️  ONLY EDIT "SYMBOL" — everything else auto-adjusts
# ══════════════════════════════════════════════════════════════


# ════════════════════════════════════════════════════════════════
# CELL 1 — Install packages (run once)
# ════════════════════════════════════════════════════════════════
# !pip install yfinance pandas numpy requests python-dotenv -q


# ════════════════════════════════════════════════════════════════
# CELL 2 — Imports, secrets, settings, all functions
# ════════════════════════════════════════════════════════════════

import yfinance as yf
import pandas as pd
import numpy as np
import requests
import time
import logging
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from IPython.display import clear_output
from google.colab import userdata
from dotenv import load_dotenv

# ── Secrets (from Colab's 🔑 Secrets panel) ─────────────────────
load_dotenv()
TOKEN   = userdata.get("TELEGRAM_TOKEN")
CHAT_ID = userdata.get("CHAT_ID")

# ════════════════════════════════════════════════════════════════
# ✏️  ONLY EDIT THIS
# ════════════════════════════════════════════════════════════════
SYMBOL            = "^NSEI"   # BTC-USD | ETH-USD | ^NSEI | AAPL | RELIANCE.NS
INTERVAL          = "5m"
MIN_PROBABILITY   = 60          # 0–100. Only enter trade if score ≥ this
ATR_SL_MULTIPLIER = 1.5         # SL = ATR × this value. Higher = wider SL
# ════════════════════════════════════════════════════════════════

IST = ZoneInfo("Asia/Kolkata")

logging.basicConfig(level=logging.WARNING)   # suppress yfinance noise


# ──────────────────────────────────────────────────────────────
# AUTO-DETECT PROFILE FROM SYMBOL
# ──────────────────────────────────────────────────────────────

def detect_profile(symbol: str) -> dict:
    s = symbol.upper()
    CRYPTO_KEYWORDS = ["BTC","ETH","BNB","SOL","XRP","DOGE","ADA","MATIC","AVAX"]

    if any(k in s for k in CRYPTO_KEYWORDS) or s.endswith("-USD"):
        price = _quick_price(symbol)
        if price and price > 10_000:  strike_round = 500
        elif price and price > 1_000: strike_round = 50
        else:                          strike_round = 1
        return dict(type="CRYPTO",    tz="UTC",              hours=None,
                    doji=0.15, strike=strike_round, max_daily_loss=3)

    if any(s.startswith(x) for x in ["^NSE","^BSE"]) or s.endswith((".NS",".BO")):
        return dict(type="INDIA",     tz="Asia/Kolkata",     hours=("09:15","15:30"),
                    doji=0.20, strike=50,            max_daily_loss=3)

    if s in ["^GSPC","^DJI","^IXIC","SPY","QQQ","IWM"]:
        return dict(type="US_INDEX",  tz="America/New_York", hours=("09:30","16:00"),
                    doji=0.20, strike=5,             max_daily_loss=3)

    return     dict(type="STOCK",     tz="America/New_York", hours=("09:30","16:00"),
                    doji=0.20, strike=1,             max_daily_loss=3)


def _quick_price(symbol: str):
    try:
        df = yf.download(symbol, period="1d", interval="1m",
                         auto_adjust=False, progress=False)
        if not df.empty:
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [c[0] for c in df.columns]
            return float(df["Close"].iloc[-1])
    except Exception:
        pass
    return None


# ──────────────────────────────────────────────────────────────
# MARKET HOURS
# ──────────────────────────────────────────────────────────────

def is_market_open(profile: dict) -> bool:
    if profile["hours"] is None:
        return True
    tz  = ZoneInfo(profile["tz"])
    now = datetime.now(tz).strftime("%H:%M")
    return profile["hours"][0] <= now <= profile["hours"][1]


# ──────────────────────────────────────────────────────────────
# TELEGRAM
# ──────────────────────────────────────────────────────────────

def ist_now() -> str:
    return datetime.now(IST).strftime("%d-%b-%Y %I:%M:%S %p IST")

def send_alert(message: str):
    if not TOKEN or not CHAT_ID:
        print("⚠️  Telegram credentials missing")
        return
    try:
        url  = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
        resp = requests.post(url, data={"chat_id": CHAT_ID, "text": message}, timeout=5)
        resp.raise_for_status()
    except Exception as e:
        print(f"⚠️  Telegram error: {e}")


# ──────────────────────────────────────────────────────────────
# DATA FETCH (with retry)
# ──────────────────────────────────────────────────────────────

def fetch_data(symbol: str, interval: str, retries: int = 3):
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(symbol, period="2d", interval=interval,
                             auto_adjust=False, progress=False)
            if not df.empty:
                return df
        except Exception as e:
            pass
        time.sleep(5 * attempt)
    return None


# ──────────────────────────────────────────────────────────────
# HEIKIN ASHI
# ──────────────────────────────────────────────────────────────

def heikin_ashi(df: pd.DataFrame):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    df = df[["Open","High","Low","Close"]].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
    if len(df) < 8:
        return None

    ha_close = (df["Open"] + df["High"] + df["Low"] + df["Close"]) / 4
    ha_open  = np.zeros(len(df))
    ha_open[0] = (df["Open"].iloc[0] + df["Close"].iloc[0]) / 2
    for i in range(1, len(df)):
        ha_open[i] = (ha_open[i-1] + ha_close.iloc[i-1]) / 2

    return pd.DataFrame({
        "open":  ha_open,
        "close": ha_close.values,
        "high":  np.maximum.reduce([ha_open, ha_close.values, df["High"].values]),
        "low":   np.minimum.reduce([ha_open, ha_close.values, df["Low"].values]),
    })


# ──────────────────────────────────────────────────────────────
# TECHNICAL INDICATORS  (RSI, ATR, EMA, Volume)
# ──────────────────────────────────────────────────────────────

def compute_indicators(df: pd.DataFrame) -> dict:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]

    close = pd.to_numeric(df["Close"], errors="coerce")
    high  = pd.to_numeric(df["High"],  errors="coerce")
    low   = pd.to_numeric(df["Low"],   errors="coerce")

    # RSI 14 — use iloc[-2] (last confirmed candle)
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rsi   = float((100 - 100 / (1 + gain / loss)).iloc[-2])

    # ATR 14
    prev_close = close.shift(1)
    tr  = pd.concat([(high - low),
                     (high - prev_close).abs(),
                     (low  - prev_close).abs()], axis=1).max(axis=1)
    atr = float(tr.rolling(14).mean().iloc[-2])

    # EMA 20
    ema = float(close.ewm(span=20, adjust=False).mean().iloc[-2])

    # Volume ratio vs 20-period average
    vol_ratio = 1.0
    if "Volume" in df.columns:
        vol = pd.to_numeric(df["Volume"], errors="coerce")
        avg = vol.rolling(20).mean().iloc[-2]
        if avg and avg > 0:
            vol_ratio = float(vol.iloc[-2] / avg)

    return {"rsi": rsi, "atr": atr, "ema": ema, "vol_ratio": vol_ratio}


# ──────────────────────────────────────────────────────────────
# PROBABILITY SCORING  (0 – 100)
# ──────────────────────────────────────────────────────────────
#
#  5 factors, each scored independently:
#
#  1. Candle Body Strength  (0-25 pts)
#     How decisive is the signal candle? body/range ratio.
#     A strong candle (big body, small wick) = more reliable.
#
#  2. RSI Zone              (0-25 pts)
#     For CALL: RSI ideally 30-60 (not overbought).
#     For PUT : RSI ideally 40-70 (not oversold).
#     Entering against RSI = low probability trade.
#
#  3. Volume Surge          (0-20 pts)
#     Volume on signal candle vs 20-period average.
#     Big moves on high volume = institutional interest.
#
#  4. Prior HA Trend        (0-20 pts)
#     How many of the last 5 candles BEFORE the flip
#     were in the opposite direction? A clean trend
#     reversal is more reliable than a choppy one.
#
#  5. Price vs EMA 20       (0-10 pts)
#     For CALL: price above EMA (uptrend context).
#     For PUT : price below EMA (downtrend context).
# ──────────────────────────────────────────────────────────────

def compute_probability(ha: pd.DataFrame, indicators: dict, direction: str) -> tuple:
    score = 0
    breakdown = {}

    curr = ha.iloc[-2]   # last confirmed signal candle

    # 1. Candle body strength
    rng   = curr["high"] - curr["low"]
    body  = abs(curr["close"] - curr["open"])
    strength = (body / rng) if rng > 0 else 0
    pts = min(round(strength * 25), 25)
    score += pts
    breakdown["Candle Strength"] = f"{pts}/25  (body={round(strength*100)}% of range)"

    # 2. RSI zone
    rsi = indicators["rsi"]
    if direction == "CALL":
        if rsi < 30:           rsi_pts = 20   # oversold → bounce likely
        elif rsi <= 60:        rsi_pts = 25   # ideal zone
        elif rsi <= 70:        rsi_pts = 10   # slightly stretched
        else:                  rsi_pts = 0    # overbought → risky
    else:
        if rsi > 70:           rsi_pts = 20
        elif rsi >= 40:        rsi_pts = 25
        elif rsi >= 30:        rsi_pts = 10
        else:                  rsi_pts = 0
    score += rsi_pts
    breakdown["RSI Zone"] = f"{rsi_pts}/25  (RSI={round(rsi, 1)})"

    # 3. Volume surge
    vr = indicators["vol_ratio"]
    if   vr >= 2.0: vol_pts = 20
    elif vr >= 1.5: vol_pts = 15
    elif vr >= 1.0: vol_pts = 8
    else:           vol_pts = 0
    score += vol_pts
    breakdown["Volume Surge"] = f"{vol_pts}/20  ({round(vr, 2)}× avg)"

    # 4. Prior HA trend (last 5 candles before signal)
    prior = ha.iloc[-7:-2]
    if direction == "CALL":
        # we want those 5 to be bearish (clean downtrend before reversal)
        agree = int((prior["close"] < prior["open"]).sum())
    else:
        agree = int((prior["close"] > prior["open"]).sum())
    trend_pts = round((agree / 5) * 20)
    score += trend_pts
    breakdown["Prior Trend"] = f"{trend_pts}/20  ({agree}/5 candles confirm reversal)"

    # 5. Price vs EMA 20
    price   = float(curr["close"])
    ema     = indicators["ema"]
    ema_pts = 10 if (direction == "CALL" and price > ema) or \
                    (direction == "PUT"  and price < ema) else 0
    score += ema_pts
    breakdown["EMA 20 Filter"] = f"{ema_pts}/10  (price={round(price,2)}, EMA={round(ema,2)})"

    return score, breakdown


# ──────────────────────────────────────────────────────────────
# SIGNAL DETECTION
# ──────────────────────────────────────────────────────────────

def not_doji(row, thresh: float) -> bool:
    rng = row["high"] - row["low"]
    return rng > 0 and abs(row["close"] - row["open"]) > thresh * rng

def check_signal(ha: pd.DataFrame, doji_thresh: float):
    """
    Uses iloc[-3] (prev) and iloc[-2] (curr) — both CONFIRMED closed candles.
    iloc[-1] is the live forming candle — intentionally ignored.
    """
    prev = ha.iloc[-3]
    curr = ha.iloc[-2]

    if (prev["close"] < prev["open"] and curr["close"] > curr["open"]
            and not_doji(prev, doji_thresh) and not_doji(curr, doji_thresh)):
        return "CALL"

    if (prev["close"] > prev["open"] and curr["close"] < curr["open"]
            and not_doji(prev, doji_thresh) and not_doji(curr, doji_thresh)):
        return "PUT"

    return None


# ──────────────────────────────────────────────────────────────
# ATR-BASED STOP LOSS  (replaces fragile candle high/low SL)
# ──────────────────────────────────────────────────────────────

def calc_sl(price: float, atr: float, direction: str) -> float:
    """
    ATR-based SL gives a volatility-adjusted buffer.
    Wider on high-volatility days → fewer false SL hits.
    Narrower on calm days → tighter risk.
    """
    if direction == "CALL":
        return price - (atr * ATR_SL_MULTIPLIER)
    else:
        return price + (atr * ATR_SL_MULTIPLIER)

def trail_sl(current_sl: float, price: float, atr: float, direction: str) -> float:
    new_sl = calc_sl(price, atr, direction)
    if direction == "CALL":
        return max(current_sl, new_sl)   # only move UP
    else:
        return min(current_sl, new_sl)   # only move DOWN


# ──────────────────────────────────────────────────────────────
# TIMING
# ──────────────────────────────────────────────────────────────

def seconds_until_next_5min() -> float:
    now       = datetime.now()
    delta     = timedelta(minutes=(5 - now.minute % 5))
    next_time = (now + delta).replace(second=5, microsecond=0)
    return max(5.0, (next_time - now).total_seconds())


# ──────────────────────────────────────────────────────────────
# COLAB LIVE DASHBOARD
# ──────────────────────────────────────────────────────────────

def print_dashboard(stats: dict, position, trailing_sl, entry_price,
                    latest_price: float, profile: dict,
                    last_signal: dict, sleep_secs: float):
    clear_output(wait=True)

    now_ist = datetime.now(IST).strftime("%d-%b-%Y  %I:%M:%S %p  IST")

    pnl      = stats["total_pnl"]
    pnl_icon = "📈" if pnl >= 0 else "📉"
    wr       = (stats["wins"] / stats["total_trades"] * 100) if stats["total_trades"] > 0 else 0

    # Unrealized P&L for open position
    unrealized = ""
    if position and entry_price:
        if position == "CALL":
            upnl = latest_price - entry_price
        else:
            upnl = entry_price - latest_price
        upnl_icon = "📈" if upnl >= 0 else "📉"
        unrealized = f"  Unrealized P&L : {upnl_icon} {round(upnl, 2)}\n"

    sl_display = f"{round(trailing_sl, 2)}" if trailing_sl else "—"

    prob_section = ""
    if last_signal:
        prob_section = (
            f"\n  ─── Last Signal ──────────────────────────\n"
            f"  Direction    : {last_signal.get('direction','—')}\n"
            f"  Probability  : {last_signal.get('score','—')}%\n"
        )
        for k, v in last_signal.get("breakdown", {}).items():
            prob_section += f"    {'·'} {k:18s}: {v}\n"
        if last_signal.get("skipped"):
            prob_section += f"  ⚠️  Trade SKIPPED (below {MIN_PROBABILITY}% threshold)\n"

    print(
        f"\n"
        f"  ╔══════════════════════════════════════════╗\n"
        f"  ║  🤖  TRADING BOT  ·  {SYMBOL:<18s}  ║\n"
        f"  ╚══════════════════════════════════════════╝\n"
        f"  🕐  {now_ist}\n"
        f"  Profile      : {profile['type']}  |  Min Prob: {MIN_PROBABILITY}%  |  ATR×{ATR_SL_MULTIPLIER}\n"
        f"\n"
        f"  ─── Market ───────────────────────────────\n"
        f"  💰 Price       : {round(latest_price, 4)}\n"
        f"  📍 Position    : {position or 'None (waiting for signal)'}\n"
        f"  🛡️  Trailing SL : {sl_display}\n"
        f"{unrealized}"
        f"\n"
        f"  ─── Statistics ───────────────────────────\n"
        f"  Entries        : {stats['total_trades']}\n"
        f"  Wins   ✅      : {stats['wins']}\n"
        f"  Losses ❌      : {stats['losses']}\n"
        f"  Skipped ⚠️     : {stats['skipped']}\n"
        f"  Win Rate       : {round(wr, 1)}%\n"
        f"  Best Trade     : {round(stats['best'], 2) if stats['best'] is not None else '—'}\n"
        f"  Worst Trade    : {round(stats['worst'], 2) if stats['worst'] is not None else '—'}\n"
        f"  {pnl_icon} Total P&L    : {round(pnl, 2)} pts\n"
        f"  Daily Losses   : {stats['daily_losses']}/{profile['max_daily_loss']}\n"
        f"{prob_section}"
        f"\n"
        f"  ⏳ Next candle check in {round(sleep_secs)}s...\n"
        f"  (Ctrl+C to stop)\n"
    )


# ════════════════════════════════════════════════════════════════
# CELL 3 — MAIN LOOP  (run this cell to start the bot)
# ════════════════════════════════════════════════════════════════

def main():
    profile = detect_profile(SYMBOL)

    print(f"🚀 Bot starting — {SYMBOL}  |  Profile: {profile['type']}")
    print(f"   Min Probability : {MIN_PROBABILITY}%")
    print(f"   ATR SL Multiplier: {ATR_SL_MULTIPLIER}×")
    print(f"   Market hours    : {profile['hours'] or '24×7'}")

    send_alert(
        f"🤖 Bot started\n"
        f"Symbol   : {SYMBOL}\n"
        f"Type     : {profile['type']}\n"
        f"Min Prob : {MIN_PROBABILITY}%\n"
        f"ATR SL   : {ATR_SL_MULTIPLIER}×\n"
        f"Time     : {ist_now()}"
    )

    # ── State ──────────────────────────────────────────────────
    position         = None
    trailing_sl_val  = None
    entry_price      = None
    last_candle_time = None
    last_signal_info = {}
    last_reset_day   = datetime.now().date()

    stats = dict(
        total_trades=0, wins=0, losses=0, skipped=0,
        total_pnl=0.0, best=None, worst=None, daily_losses=0
    )

    # ── Loop ───────────────────────────────────────────────────
    while True:

        # Reset daily losses at midnight
        today = datetime.now().date()
        if today != last_reset_day:
            stats["daily_losses"] = 0
            last_reset_day = today

        # Daily loss limit
        if stats["daily_losses"] >= profile["max_daily_loss"]:
            sleep_secs = seconds_until_next_5min()
            print_dashboard(stats, position, trailing_sl_val, entry_price,
                            0, profile, last_signal_info, sleep_secs)
            print("  ⛔ Daily loss limit reached — paused till midnight")
            time.sleep(60)
            continue

        # Market hours
        if not is_market_open(profile):
            print(f"  🕐 Market closed — checking again in 60s")
            time.sleep(60)
            continue

        # Fetch data
        df = fetch_data(SYMBOL, INTERVAL)
        if df is None:
            print("  ⚠️  Data fetch failed — retrying in 30s")
            time.sleep(30)
            continue

        # Deduplicate candles
        candle_time = df.index[-1]
        if candle_time == last_candle_time:
            time.sleep(5)
            continue
        last_candle_time = candle_time

        # Build HA + indicators
        ha = heikin_ashi(df)
        if ha is None:
            time.sleep(10)
            continue

        try:
            indicators = compute_indicators(df)
        except Exception:
            time.sleep(10)
            continue

        latest_price = float(
            df["Close"].iloc[-1].item()
            if hasattr(df["Close"].iloc[-1], "item")
            else df["Close"].iloc[-1]
        )
        atr = indicators["atr"]

        signal = check_signal(ha, profile["doji"])

        # ── ENTRY ────────────────────────────────────────────
        if signal and position is None:
            score, breakdown = compute_probability(ha, indicators, signal)

            last_signal_info = {
                "direction": signal,
                "score":     score,
                "breakdown": breakdown,
                "skipped":   score < MIN_PROBABILITY
            }

            if score < MIN_PROBABILITY:
                # Low probability — skip this trade
                stats["skipped"] += 1
            else:
                position        = signal
                entry_price     = latest_price
                trailing_sl_val = calc_sl(latest_price, atr, signal)
                atm             = round(latest_price / profile["strike"]) * profile["strike"]
                stats["total_trades"] += 1

                msg = (
                    f"📊 {SYMBOL}  —  {signal} ENTRY\n"
                    f"{'─'*30}\n"
                    f"Probability : {score}%\n"
                    f"Spot Price  : {round(latest_price, 2)}\n"
                    f"ATM Strike  : {atm}\n"
                    f"Stop Loss   : {round(trailing_sl_val, 2)}\n"
                    f"ATR         : {round(atr, 2)}\n"
                    f"Time (IST)  : {ist_now()}\n"
                    f"{'─'*30}\n"
                    + "\n".join(f"  {k}: {v}" for k, v in breakdown.items())
                )
                send_alert(msg)

        # ── TRAILING + EXIT ───────────────────────────────────
        elif position == "CALL" and trailing_sl_val is not None:
            trailing_sl_val = trail_sl(trailing_sl_val, latest_price, atr, "CALL")

            if latest_price < trailing_sl_val:
                pnl = latest_price - entry_price
                stats["losses"]       += 1
                stats["daily_losses"] += 1
                stats["total_pnl"]    += pnl
                stats["worst"] = pnl if stats["worst"] is None else min(stats["worst"], pnl)

                send_alert(
                    f"❌ CALL SL HIT  —  {SYMBOL}\n"
                    f"Entry : {round(entry_price, 2)}\n"
                    f"Exit  : {round(latest_price, 2)}\n"
                    f"P&L   : {round(pnl, 2)} pts\n"
                    f"Time  : {ist_now()}"
                )
                position, trailing_sl_val, entry_price = None, None, None

        elif position == "PUT" and trailing_sl_val is not None:
            trailing_sl_val = trail_sl(trailing_sl_val, latest_price, atr, "PUT")

            if latest_price > trailing_sl_val:
                pnl = entry_price - latest_price
                stats["losses"]       += 1
                stats["daily_losses"] += 1
                stats["total_pnl"]    += pnl
                stats["worst"] = pnl if stats["worst"] is None else min(stats["worst"], pnl)

                send_alert(
                    f"❌ PUT SL HIT  —  {SYMBOL}\n"
                    f"Entry : {round(entry_price, 2)}\n"
                    f"Exit  : {round(latest_price, 2)}\n"
                    f"P&L   : {round(pnl, 2)} pts\n"
                    f"Time  : {ist_now()}"
                )
                position, trailing_sl_val, entry_price = None, None, None

        # ── Update best trade (track P&L while position open) ─
        if position and entry_price:
            unrealized = (latest_price - entry_price) if position == "CALL" \
                    else (entry_price - latest_price)
            stats["best"] = unrealized if stats["best"] is None \
                       else max(stats["best"], unrealized)

        # ── Dashboard ─────────────────────────────────────────
        sleep_secs = seconds_until_next_5min()
        print_dashboard(stats, position, trailing_sl_val, entry_price,
                        latest_price, profile, last_signal_info, sleep_secs)

        time.sleep(sleep_secs)


# Run
main()

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import time
import logging
from datetime import datetime, timedelta

from zoneinfo import ZoneInfo
from IPython.display import clear_output
from google.colab import userdata

# --- 1. SETTINGS & ASSETS ---
# Expanded list with popular symbols from different markets
SYMBOLS = [
    "BTC-USD", "ETH-USD", "SOL-USD",  # Crypto
    "^NSEI", "RELIANCE.NS", "HDFCBANK.NS", # India (Nifty 50, Reliance, HDFC)
    "AAPL", "TSLA", "NVDA", "MSFT"      # US (Apple, Tesla, Nvidia, Microsoft)
]

INTERVAL = "5m"
MIN_PROBABILITY = 60
ATR_SL_MULTIPLIER = 1.5
TP_THRESHOLD = 0.05  # 5% Fixed Take Profit
IST = ZoneInfo("Asia/Kolkata")

TOKEN = userdata.get("TELEGRAM_TOKEN")
CHAT_ID = userdata.get("CHAT_ID")

logging.basicConfig(level=logging.WARNING)

# --- 2. CORE FUNCTIONS ---
def detect_profile(symbol: str) -> dict:
    s = symbol.upper()
    if "BTC" in s or "ETH" in s or "SOL" in s or s.endswith("-USD"):
        return dict(type="CRYPTO", tz="UTC", hours=None, doji=0.15, strike=1)
    if "^NSE" in s or s.endswith(".NS"):
        return dict(type="INDIA", tz="Asia/Kolkata", hours=("09:15","15:30"), doji=0.20, strike=50)
    return dict(type="US_STOCK", tz="America/New_York", hours=("09:30","16:00"), doji=0.20, strike=1)

def is_market_open(profile: dict) -> bool:
    if profile["hours"] is None: return True
    now_tz = datetime.now(ZoneInfo(profile["tz"]))
    if now_tz.weekday() >= 5: return False
    curr = now_tz.strftime("%H:%M")
    return profile["hours"][0] <= curr <= profile["hours"][1]

def send_alert(msg):
    if not TOKEN or not CHAT_ID: print(f"[LOG] {msg}"); return
    try: requests.post(f"https://api.telegram.org/bot{TOKEN}/sendMessage", data={"chat_id": CHAT_ID, "text": msg}, timeout=5)
    except: pass

def fetch_data(symbol, interval):
    try:
        df = yf.download(symbol, period="2d", interval=interval, auto_adjust=False, progress=False)
        if isinstance(df.columns, pd.MultiIndex): df.columns = [c[0] for c in df.columns]
        return df if not df.empty else None
    except: return None

def heikin_ashi(df):
    df = df[["Open","High","Low","Close"]].apply(pd.to_numeric).dropna().reset_index(drop=True)
    if len(df) < 10: return None
    ha_close = (df["Open"] + df["High"] + df["Low"] + df["Close"]) / 4
    ha_open = np.zeros(len(df))
    ha_open[0] = (df["Open"].iloc[0] + df["Close"].iloc[0]) / 2
    for i in range(1, len(df)): ha_open[i] = (ha_open[i-1] + ha_close.iloc[i-1]) / 2
    return pd.DataFrame({"open": ha_open, "close": ha_close.values, "high": np.maximum.reduce([ha_open, ha_close.values, df["High"].values]), "low": np.minimum.reduce([ha_open, ha_close.values, df["Low"].values])})

def compute_indicators(df):
    close, high, low = df["Close"], df["High"], df["Low"]
    delta = close.diff(); gain = delta.clip(lower=0).rolling(14).mean(); loss = (-delta.clip(upper=0)).rolling(14).mean()
    rsi = float((100 - 100 / (1 + gain / loss)).iloc[-2])
    tr = pd.concat([(high - low), (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    return {"rsi": rsi, "atr": float(tr.rolling(14).mean().iloc[-2]), "ema": float(close.ewm(span=20).mean().iloc[-2])}

def print_dashboard(symbol_states):
    clear_output(wait=True)
    print(f"\n  === MULTI-SYMBOL BOT ({datetime.now(IST).strftime('%I:%M:%S %p')}) ===")
    print(f"  {'SYMBOL':<12} {'STATUS':<8} {'POS':<6} {'ENTRY':<10} {'PRICE':<10} {'UNREAL':<10}")
    print("  " + "-"*65)
    g_trades, g_pnl = 0, 0.0
    for s, st in symbol_states.items():
        m_stat = "OPEN" if is_market_open(st['profile']) else "CLOSED"
        pos, ent, cur = st['position'] or "-", round(st['entry_price'] or 0, 2), round(st['latest_price'] or 0, 2)
        unreal = 0.0
        if pos == "CALL": unreal = cur - ent
        elif pos == "PUT": unreal = ent - cur
        print(f"  {s:<12} {m_stat:<8} {pos:<6} {ent:<10} {cur:<10} {round(unreal,2):<10}")
        g_trades += st['stats']['trades']; g_pnl += st['stats']['pnl']
    print(f"\n  Global Trades: {g_trades} | Realized P&L: {round(g_pnl, 2)} pts")

# --- 3. MAIN LOOP ---
def start_bot():
    states = {s: {"position": None, "entry_price": None, "trailing_sl": None, "latest_price": 0,
                  "profile": detect_profile(s), "stats": {"trades":0, "pnl":0.0}, "last_time": None} for s in SYMBOLS}

    while True:
        for s in SYMBOLS:
            st = states[s]
            if not is_market_open(st["profile"]): continue

            df = fetch_data(s, INTERVAL)
            if df is None: continue
            if df.index[-1] == st["last_time"]: continue
            st["last_time"] = df.index[-1]

            ha = heikin_ashi(df); ind = compute_indicators(df)
            price = float(df["Close"].iloc[-1]); st["latest_price"] = price

            # Exit Logic (TP & SL)
            if st["position"]:
                p_pct = (price - st["entry_price"])/st["entry_price"] if st["position"] == "CALL" else (st["entry_price"] - price)/st["entry_price"]
                if p_pct >= TP_THRESHOLD or (st["position"] == "CALL" and price < st["trailing_sl"]) or (st["position"] == "PUT" and price > st["trailing_sl"]):
                    pnl = (price - st["entry_price"]) if st["position"] == "CALL" else (st["entry_price"] - price)
                    st["stats"]["pnl"] += pnl; send_alert(f"EXIT {s} | P&L: {round(pnl,2)}")
                    st["position"] = None; st["entry_price"] = None

            # Entry Logic (Simplified Signal)
            elif ha.iloc[-2]["close"] > ha.iloc[-2]["open"] and ha.iloc[-3]["close"] < ha.iloc[-3]["open"]:
                st["position"], st["entry_price"] = "CALL", price
                st["trailing_sl"] = price - (ind["atr"] * ATR_SL_MULTIPLIER); st["stats"]["trades"] += 1
                send_alert(f"ENTRY CALL {s} at {price}")

        print_dashboard(states)
        time.sleep(30)

start_bot()


  === MULTI-SYMBOL BOT (02:21:13 PM) ===
  SYMBOL       STATUS   POS    ENTRY      PRICE      UNREAL    
  -----------------------------------------------------------------
  BTC-USD      OPEN     CALL   67865.47   67908.8    43.33     
  ETH-USD      OPEN     -      0          1981.93    0.0       
  SOL-USD      OPEN     CALL   84.33      84.48      0.15      
  ^NSEI        CLOSED   -      0          0          0.0       
  RELIANCE.NS  CLOSED   -      0          0          0.0       
  HDFCBANK.NS  CLOSED   -      0          0          0.0       
  AAPL         CLOSED   -      0          0          0.0       
  TSLA         CLOSED   -      0          0          0.0       
  NVDA         CLOSED   -      0          0          0.0       
  MSFT         CLOSED   -      0          0          0.0       

  Global Trades: 2 | Realized P&L: 0.0 pts
